# Flash Attention Benchmark

Companion notebook for the [Efficient Attention](https://codemia.ai/courses/llm_inside_and_out/efficient_attention) lesson on Codemia.

**What you will build**: a head-to-head benchmark of the textbook O(T^2)-memory attention versus Flash Attention (via `torch.nn.functional.scaled_dot_product_attention`). Vary the sequence length from 512 to 16384, measure wall-clock time and peak memory, verify numerical equivalence, and plot the scaling curves.

**Karpathy companion**: [Let's reproduce GPT-2 (124M)](https://www.youtube.com/watch?v=l8pRSuU81PU) switches nanoGPT over to Flash Attention and explains the speedup. Adapted in part from karpathy/build-nanogpt (MIT).

**Time**: ~20-40 minutes

**GPU required**: yes (T4 or better). The naive path will OOM on long sequences; we detect that and skip gracefully.

## Setup

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import math
import time
import gc
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "This notebook needs a GPU. On Colab: Runtime -> Change runtime type -> T4 GPU."
device = 'cuda'
torch.manual_seed(0)

# Print the GPU we got -- useful when debugging why timings look different from the reference.
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Concept recap

Standard attention materializes the full `T x T` attention matrix (softmax weights) in HBM. At 16k tokens that's 256 million floats per head per layer, and the majority of time goes to *reading and writing* it, not computing it. Flash Attention (Dao et al., 2022) never materializes the full matrix: it tiles queries and keys into blocks that fit in SRAM, streams through them, and accumulates the softmax online using a numerically stable running-max trick. Result: O(T) memory, O(T^2) compute, and usually 2-5x faster wall-clock because it's memory-bandwidth bound.

PyTorch ships Flash Attention inside `F.scaled_dot_product_attention` since 2.0. On an Ampere/Hopper GPU it dispatches to FlashAttention-2 automatically.

## Implementation: the naive reference attention

In [ ]:
def naive_attention(q, k, v, causal=True):
    # q, k, v: (B, H, T, D)
    B, H, T, D = q.shape
    # The gigantic (T, T) score matrix lives in HBM here. At long T this is where we OOM.
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(D)
    if causal:
        # Materialize the mask on-GPU; torch.triu is cheap.
        mask = torch.triu(torch.full((T, T), float('-inf'), device=q.device), diagonal=1)
        scores = scores + mask
    attn = torch.softmax(scores, dim=-1)
    return attn @ v

## Implementation: Flash path via `F.scaled_dot_product_attention`

Nothing to implement -- it's one function call. But we wrap it for symmetry with the naive version, and tell PyTorch to strictly use the flash kernel so the benchmark is apples-to-apples.

In [ ]:
def flash_attention(q, k, v, causal=True):
    # The is_causal flag asks the kernel to apply a causal mask without materializing it.
    # That's half the memory savings right there.
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal)

## Numerical equivalence check

In [ ]:
# Small tensors so we can run both paths and diff them. Use fp32 for a fair comparison;
# flash on fp16 introduces ~1e-3 noise that is fine in training but would confuse this check.
B, H, T, D = 1, 4, 256, 64
q = torch.randn(B, H, T, D, device=device)
k = torch.randn(B, H, T, D, device=device)
v = torch.randn(B, H, T, D, device=device)

out_naive = naive_attention(q, k, v, causal=True)
# Force the kernel backend to the math path for the reference, then flash for comparison.
with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):
    out_flash = flash_attention(q, k, v, causal=True)

max_err = (out_naive - out_flash).abs().max().item()
print(f"max abs diff: {max_err:.2e}")
print("passed" if max_err < 1e-4 else "FAILED -- investigate")

## Benchmark harness

We time each kernel with CUDA events (not `time.time` -- Python clocks don't see GPU work) and record peak memory with `torch.cuda.max_memory_allocated`. Both are reset before each measurement.

In [ ]:
def benchmark(fn, q, k, v, n_warmup=3, n_iter=10):
    # Warm up first: the first call includes CUDA graph / autotuner overhead.
    for _ in range(n_warmup):
        fn(q, k, v)
    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(n_iter):
        fn(q, k, v)
    end.record()
    torch.cuda.synchronize()

    ms_per_iter = start.elapsed_time(end) / n_iter
    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    return ms_per_iter, peak_mb

In [ ]:
# Benchmark settings chosen so they fit on a T4 (16 GB) for flash at all lengths,
# and fit for naive up to ~4096 tokens. The naive path will OOM at 8192 on a T4; we catch it.
B, H, D = 1, 8, 64
dtype = torch.float16  # flash shines in fp16/bf16. Use that for the throughput test.
seq_lens = [512, 1024, 2048, 4096, 8192, 16384]

results = {"naive": {}, "flash": {}}
for T in seq_lens:
    q = torch.randn(B, H, T, D, device=device, dtype=dtype)
    k = torch.randn(B, H, T, D, device=device, dtype=dtype)
    v = torch.randn(B, H, T, D, device=device, dtype=dtype)

    # Naive path: may OOM at long sequences. Try, catch, report.
    try:
        torch.cuda.empty_cache(); gc.collect()
        ms, mem = benchmark(lambda q, k, v: naive_attention(q, k, v), q, k, v)
        results["naive"][T] = (ms, mem)
        print(f"T={T:5d} naive: {ms:8.2f} ms  peak {mem/1000:6.2f} GB")
    except torch.cuda.OutOfMemoryError:
        results["naive"][T] = None
        print(f"T={T:5d} naive: OOM")
        torch.cuda.empty_cache()

    # Flash path: should comfortably handle all lengths on T4.
    torch.cuda.empty_cache(); gc.collect()
    with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):
        ms, mem = benchmark(lambda q, k, v: flash_attention(q, k, v), q, k, v)
    results["flash"][T] = (ms, mem)
    print(f"T={T:5d} flash: {ms:8.2f} ms  peak {mem/1000:6.2f} GB")
    del q, k, v

## Plot the scaling curves

In [ ]:
def unpack(d, idx):
    # Pull out time (idx=0) or memory (idx=1); None entries mean OOM and are skipped.
    xs, ys = [], []
    for T, val in d.items():
        if val is not None:
            xs.append(T); ys.append(val[idx])
    return xs, ys

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Time. Expect O(T^2) for both, but flash with a much smaller constant.
ax = axes[0]
xs, ys = unpack(results["naive"], 0); ax.plot(xs, ys, 'o-', label='naive')
xs, ys = unpack(results["flash"], 0); ax.plot(xs, ys, 's-', label='flash')
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('sequence length'); ax.set_ylabel('ms per forward')
ax.set_title('Wall-clock time')
ax.legend()

# Memory. Naive grows O(T^2) because of the full scores matrix.
# Flash grows O(T) because it never allocates that matrix.
ax = axes[1]
xs, ys = unpack(results["naive"], 1); ax.plot(xs, ys, 'o-', label='naive')
xs, ys = unpack(results["flash"], 1); ax.plot(xs, ys, 's-', label='flash')
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('sequence length'); ax.set_ylabel('peak MB')
ax.set_title('Peak GPU memory')
ax.legend()
plt.tight_layout()
plt.show()

## Variation 1: speedup ratio

The interesting number is not absolute time but the ratio. At short sequences (T=512) both kernels are compute-bound and flash is only slightly faster. As T grows, naive becomes memory-bandwidth bound and flash pulls ahead.

In [ ]:
print(f"{'T':>6}  {'naive ms':>10}  {'flash ms':>10}  {'speedup':>10}")
for T in seq_lens:
    n, f = results['naive'][T], results['flash'][T]
    if n is None:
        print(f"{T:>6}  {'OOM':>10}  {f[0]:>10.2f}  {'inf':>10}")
    else:
        print(f"{T:>6}  {n[0]:>10.2f}  {f[0]:>10.2f}  {n[0]/f[0]:>9.2f}x")

## Variation 2: backward pass

The gap is even larger on the backward pass because flash recomputes the attention matrix from its stored running statistics instead of storing the full matrix for gradients. Quick check: time one forward+backward at a middle sequence length.

In [ ]:
def fwd_bwd(fn, q, k, v):
    q = q.detach().requires_grad_(True)
    k = k.detach().requires_grad_(True)
    v = v.detach().requires_grad_(True)
    out = fn(q, k, v)
    out.sum().backward()

T = 2048
q = torch.randn(1, 8, T, 64, device=device, dtype=torch.float16)
k = torch.randn(1, 8, T, 64, device=device, dtype=torch.float16)
v = torch.randn(1, 8, T, 64, device=device, dtype=torch.float16)

ms_n, mem_n = benchmark(lambda q, k, v: fwd_bwd(naive_attention, q, k, v), q, k, v, n_iter=5)
with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):
    ms_f, mem_f = benchmark(lambda q, k, v: fwd_bwd(flash_attention, q, k, v), q, k, v, n_iter=5)

print(f"fwd+bwd at T={T}")
print(f"  naive: {ms_n:6.2f} ms, peak {mem_n/1000:.2f} GB")
print(f"  flash: {ms_f:6.2f} ms, peak {mem_f/1000:.2f} GB")
print(f"  speedup: {ms_n/ms_f:.2f}x, memory ratio: {mem_n/mem_f:.2f}x")

## What to try next

- Add mem-efficient attention (`enable_flash=False, enable_mem_efficient=True`) as a third column. It's what flash falls back to on older GPUs.
- Run the benchmark with bf16 and fp32 and see how much the speedup depends on dtype.
- Swap in a real model (GPT-2 small) and measure end-to-end training step time naive vs flash on the same batch.
- Read [FlashAttention-2](https://arxiv.org/abs/2307.08691) and identify which optimization (parallelization along the sequence axis) gives most of the speedup over FlashAttention-1.